# Tarea · ¿Cuánto ayuda reducir los contactos?

## Una intervención en una comunidad

En una comunidad de $N=10,000$ personas aparecen 10 casos infecciosos. Para este ejercicio usamos el modelo SIR con $\beta=0.40$ por día y $\gamma=0.25$ por día. Una campaña busca reducir los contactos mediante cambios temporales en las actividades comunitarias.

**Tu misión:** comparar un escenario sin intervención con otro donde los contactos se reducen un **40 % desde el día 0** y la reducción se mantiene durante toda la simulación.



### Instrucciones

1. Ejecuta las celdas en orden. Necesitas `numpy`, `scipy` y `matplotlib`, como en clase.
2. Reemplaza cada `None` señalado con `TODO` por el valor o la expresión correspondiente. Los `None` son espacios por completar, no respuestas.
3. Completa las respuestas escritas y conserva las gráficas y los resultados de tu ejecución.


## 1. Antes de programar: predice

Supongamos que $\beta=cp$, donde $c$ representa la frecuencia de contactos y $p$ la probabilidad de transmisión por contacto. Si reducimos los contactos en una proporción $q$, manteniendo lo demás constante,

$$
\beta_{\mathrm{intervención}}=(1-q)\beta.
$$

Aquí $q=0.40$ significa una reducción del 40 %, no del 0.40 %.

**Completa sin ejecutar todavía la simulación:**

- Sin intervención, $R_0=\beta/\gamma=$ …
- Con intervención, $\beta_{\mathrm{intervención}}=$ … .
- Con intervención, $R_{0,\mathrm{intervención}}=\beta_{\mathrm{intervención}}/\gamma=$ …
- ¿Esperas que las personas infecciosas aumenten al inicio en cada escenario? ¿Por qué? …

Puedes justificar el crecimiento inicial directamente con $I'(0)=I(0)[\beta S(0)/N-\gamma]$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

N = 10_000
I0 = 10
R_init = 0
S0 = N - I0 - R_init
y0 = [S0, I0, R_init]

beta = 0.40
gamma = 0.25
t_eval = np.linspace(0, 200, 1001)

q = None                 # TODO: reducción del 40 %, expresada como proporción
beta_intervencion = None # TODO: usa q y beta

print(f"R₀ sin intervención: {beta / gamma:.2f}")
print(f"R₀ con intervención: {beta_intervencion / gamma:.2f}")


## 2. Completa las ecuaciones

Usamos cantidades de personas, no proporciones:

$$
S'=-\beta\frac{SI}{N},\qquad
I'=\beta\frac{SI}{N}-\gamma I,\qquad
R'=\gamma I.
$$

Completa las dos tasas y las tres derivadas. La función recibe `beta` y `gamma` como argumentos para poder reutilizarla en ambos escenarios.


In [ ]:
def sir(t, y, beta, gamma):
    S, I, R = y
    nuevas_infecciones = None  # TODO
    recuperaciones = None     # TODO
    dS = None                 # TODO
    dI = None                 # TODO
    dR = None                 # TODO
    return [dS, dI, dR]

# Comprobación: la suma de las derivadas debe ser cero.
derivadas = sir(0, y0, beta, gamma)
print("Derivadas iniciales:", derivadas)
print("Suma de las derivadas:", sum(derivadas))


## 3. Simula los dos escenarios

La solución del primer escenario esta completa `sol_base`. En la segunda utiliza el mismo intervalo, las mismas condiciones iniciales y los mismos tiempos de salida; cambia solamente la tasa de transmisión.


In [ ]:
sol_base = solve_ivp(
    sir, [t_eval[0], t_eval[-1]], y0,
    t_eval=t_eval, args=(beta, gamma), rtol=1e-8, atol=1e-8
)

sol_intervencion = solve_ivp(
    sir, [t_eval[0], t_eval[-1]], y0,
    t_eval=t_eval,
    args=(None, gamma),  # TODO: sustituye None por la tasa con intervención
    rtol=1e-8, atol=1e-8
)

print("Integraciones exitosas:", sol_base.success, sol_intervencion.success)
S_base, I_base, R_base = sol_base.y
S_intervencion, I_intervencion, R_intervencion = sol_intervencion.y

print("Mayor error en S + I + R = N:")
print("Sin intervención:", np.max(np.abs(S_base + I_base + R_base - N)))
print("Con intervención:", np.max(np.abs(S_intervencion + I_intervencion + R_intervencion - N)))


## 4. Compara las curvas

Completa la curva con intervención. Incluye en la leyenda qué escenario representa cada curva.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sol_base.t, I_base, label="Sin intervención")
ax.plot(sol_intervencion.t, None, label="Contactos reducidos un 40 %")  # TODO
ax.set(xlabel="Tiempo (días)", ylabel="Personas infecciosas",
       title="Efecto de reducir los contactos", xlim=(0, 200))
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 5. Del gráfico a los números

Calcula para cada escenario:

- El mayor número de personas infecciosas y el día en que ocurre.
- Las personas que se han infectado hasta el día 200.

`np.argmax(I)` devuelve el índice del máximo. Como $R(0)=0$, las infecciones acumuladas hasta un tiempo $T$, incluyendo los 10 casos iniciales, son

$$
N-S(T)=I(T)+R(T).
$$

Esto no es necesariamente el tamaño final del brote: revisa cuántas personas siguen infecciosas en el último día. Si el máximo está en el día 0, significa que el número de infecciosos disminuye desde el inicio.


In [ ]:
def resumir(nombre, sol):
    S, I, R = sol.y
    indice_pico = None  # TODO: índice del máximo de I
    acumuladas = None  # TODO: personas que se han infectado hasta el último día
    print(nombre)
    print(f"  Pico: {I[indice_pico]:.1f} personas")
    print(f"  Día del pico: {sol.t[indice_pico]:.1f}")
    print(f"  Infecciones acumuladas: {acumuladas:.1f}")
    print(f"  Infecciosos en el último día: {I[-1]:.4f}")

resumir("Sin intervención", sol_base)
resumir("Con intervención", sol_intervencion)


### Interpreta tus resultados

1. ¿Se cumplieron tus predicciones? Apoya tu respuesta con dos resultados numéricos.

   **Respuesta:** …

2. ¿Cuántas infecciones acumuladas menos hay al día 200 con la intervención? ¿Qué porcentaje representan respecto al escenario sin intervención?

   **Respuesta:** …

3. ¿Por qué puede disminuir $I$ aunque aún queden muchas personas susceptibles?

   **Respuesta:** …

4. En este modelo, reducir los contactos un 40 % ¿produce necesariamente un 40 % menos de infecciones acumuladas? Usa tus resultados para responder.

   **Respuesta:** …

5. Menciona un supuesto que podría hacer que el resultado en una comunidad real fuera diferente.

   **Respuesta:** …

## 6. ¿Qué reducción sería suficiente?

Buscamos una reducción que haga que el número reproductivo básico del escenario intervenido sea menor que uno:

$$
\frac{(1-q)\beta}{\gamma}<1.
$$

Despeja $q$ y calcula el umbral para los parámetros del ejercicio. Este criterio garantiza que $I$ disminuya desde el inicio incluso si casi toda la población es susceptible; la condición exacta con las condiciones iniciales dadas es $(1-q)\beta S(0)/N<\gamma$.

**Desarrollo y respuesta:** …

Escoge una reducción justo por debajo y otra por encima del umbral. Simula ambas desde el día 0 y compara la dirección del cambio inicial. En el valor exacto del umbral, ¿qué implica que $S(0)<N$?


In [ ]:
# TODO: escribe aquí tus dos experimentos y muestra una comparación.


## Reto opcional · La intervención llega tarde

En los ejercicios anteriores, la reducción empezaba en el día 0. Ahora mantén $\beta=0.40$ hasta el día 20 y aplica la reducción del 40 % a partir de ese momento, de forma permanente.

**¿Qué cambia si la misma medida se aplica más tarde?**

Resuelve el problema en dos etapas:

1. Integra el escenario sin intervención desde el día 0 hasta el día 20.
2. Usa el estado al final de esa etapa como condición inicial de la segunda.
3. Integra desde el día 20 hasta el día 200 con la tasa reducida.
4. Une los resultados y compara tres escenarios: sin intervención, intervención desde el día 0 e intervención desde el día 20.

**Pistas:** `sol.y[:, -1]` selecciona las tres variables en el último tiempo; `np.concatenate` une arreglos. Evita repetir el día 20 al unir las etapas. Conserva $N$ durante ambas etapas: la medida no reinicia la epidemia ni cambia la población.


In [ ]:
dia_intervencion = 20

# TODO: primera integración, con beta original, hasta el día 20.

# TODO: extraer el último estado como condición inicial de la segunda etapa.

# TODO: segunda integración, con beta_intervencion, desde el día 20.

# TODO: unir los tiempos y las curvas de I y graficar los tres escenarios.


### Para discutir el reto

- ¿La intervención tardía logra el mismo efecto que la temprana? Compara el pico y las infecciones acumuladas al día 200.
- ¿Qué ocurre con la pendiente de I cuando empieza la intervención? ¿El número de infecciosos cambia de golpe o cambia su tendencia?
- **Un paso más:** prueba inicios en los días 10, 20 y 40. ¿Cuál es el último de esos días que mantiene el pico por debajo de 100 personas infecciosas? Si ninguno lo logra, indícalo. Justifica con tus simulaciones.

**Conclusión final (3–5 frases):** ¿qué aprendiste sobre la intensidad y el momento de una intervención, dentro de los supuestos de este modelo?

…
